# 第 26 天：ML因子1

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：ML因子1
> 必做：特征工程
> 选做：因子扩展
> 目标产出：训练数据集

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 把因子矩阵整理成机器学习训练表。
2. 构造交互特征、行业特征、滚动特征。
3. 按时间切分训练集和测试集，避免未来函数。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

机器学习不是魔法锅，不是把数据倒进去就会吐出 Alpha。它更像一个放大器：特征做得好，它放大结构；特征做得乱，它放大噪声。

## 5. 今日核心实验


### 实验 1：把因子矩阵变成机器学习训练表

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
feature_names = ["value", "quality", "growth", "momentum_20", "momentum_60", "low_vol", "liquidity", "reversal_5", "price_volume"]
ml_library = {name: factor_library[name] for name in feature_names}
panel = build_panel(ml_library, future_5d)

print("训练表形状：", panel.shape)
print(panel.head().round(4))


### 实验 2：添加交互特征：让模型看见非线性结构

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
panel_fe = panel.copy()
panel_fe["value_x_quality"] = panel_fe["value"] * panel_fe["quality"]
panel_fe["momentum_x_liquidity"] = panel_fe["momentum_20"] * panel_fe["liquidity"]
panel_fe["low_vol_x_quality"] = panel_fe["low_vol"] * panel_fe["quality"]
panel_fe["short_long_mom_gap"] = panel_fe["momentum_20"] - panel_fe["momentum_60"]

print(panel_fe[["value_x_quality", "momentum_x_liquidity", "short_long_mom_gap", "label"]].describe().round(4))


### 实验 3：添加行业特征：分类信息也要进训练集

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
asset_index = panel_fe.index.get_level_values("asset")
industry_onehot = pd.get_dummies(industries.reindex(asset_index).to_numpy(), prefix="industry")
industry_onehot.index = panel_fe.index
panel_fe = pd.concat([panel_fe, industry_onehot], axis=1)

print("加入行业后特征数量：", panel_fe.shape[1] - 1)
print(panel_fe.filter(like="industry_").head())


### 实验 4：时间切分：不能随机打散金融时间

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
train, test, split_date = time_split_panel(panel_fe, split_ratio=0.7)
features = [c for c in panel_fe.columns if c != "label"]

X_train = train[features].to_numpy()
y_train = train["label"].to_numpy()
X_test = test[features].to_numpy()
y_test = test["label"].to_numpy()

print({
    "split_date": str(split_date.date()),
    "train_rows": len(train),
    "test_rows": len(test),
    "features": len(features),
})


### 实验 5：训练数据体检：缺失、漂移、标签分布

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
health = pd.DataFrame({
    "train_mean": train[features].mean(),
    "test_mean": test[features].mean(),
    "mean_shift": test[features].mean() - train[features].mean(),
    "missing": panel_fe[features].isna().mean(),
}).sort_values("mean_shift", key=lambda s: s.abs(), ascending=False)

print(health.head(12).round(4))
print("\n标签分布：")
print(pd.Series({
    "train_label_mean": y_train.mean(),
    "test_label_mean": y_test.mean(),
    "train_label_std": y_train.std(),
    "test_label_std": y_test.std(),
}).round(5))


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：ML因子1
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：随机切分样本，导致未来数据泄露到训练集。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：把标签相关信息做成特征，造成看起来很强的假模型。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：忽略横截面标准化，模型只学到市值或价格量纲。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：特征越多越好，最后样本外反而更差。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 27 天会训练模型，并用特征重要性解释它到底学到了什么。

## 13. 一句话收尾

ML因子1 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒（更新版）

今天如果只记住一件事，记住这个：**用时间序列交叉验证替代随机K-Fold，用Purged Gap防止标签泄露。** 这是金融ML和通用ML之间最重要的方法论差异。

---

## 15. 进阶：XGBoost / LightGBM 实战（替代 Ridge 基线）

> Ridge 回归是好基线，但树模型能捕捉非线性结构和特征交互。真实量化研究中，XGBoost/LightGBM 是更主流的选择。

### 15.1 安装


In [ ]:
pip install xgboost lightgbm


### 15.2 LightGBM 训练（带早停）


In [ ]:
try:
    import lightgbm as lgb
    
    # LightGBM 对特征名中的特殊字符敏感，清理列名
    clean_features = [f"f_{i}" for i in range(len(features))]
    dtrain = lgb.Dataset(X_train, label=y_train, feature_name=clean_features)
    dvalid = lgb.Dataset(X_test, label=y_test, feature_name=clean_features, reference=dtrain)
    
    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbose": -1,
        "seed": 42,
    }
    
    lgb_model = lgb.train(
        params, dtrain,
        num_boost_round=500,
        valid_sets=[dtrain, dvalid],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
    )
    
    pred_lgb = lgb_model.predict(X_test)
    lgb_ic = pd.Series(pred_lgb).corr(pd.Series(y_test), method="spearman")
    print(f"LightGBM 样本外 Rank IC: {lgb_ic:.4f}")
    
    # 特征重要性
    importance_df = pd.DataFrame({
        "feature": clean_features,
        "importance": lgb_model.feature_importance(importance_type="gain"),
    }).sort_values("importance", ascending=False)
    print(importance_df.head(10))
    
except ImportError:
    print("LightGBM 未安装。安装命令: pip install lightgbm")


### 15.3 XGBoost 训练


In [ ]:
try:
    import xgboost as xgb
    
    dtrain_xgb = xgb.DMatrix(X_train, label=y_train)
    dtest_xgb = xgb.DMatrix(X_test, label=y_test)
    
    params_xgb = {
        "objective": "reg:squarederror",
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "seed": 42,
        "verbosity": 0,
    }
    
    xgb_model = xgb.train(
        params_xgb, dtrain_xgb,
        num_boost_round=500,
        evals=[(dtrain_xgb, "train"), (dtest_xgb, "eval")],
        early_stopping_rounds=50,
        verbose_eval=False,
    )
    
    pred_xgb = xgb_model.predict(dtest_xgb)
    xgb_ic = pd.Series(pred_xgb).corr(pd.Series(y_test), method="spearman")
    print(f"XGBoost 样本外 Rank IC: {xgb_ic:.4f}")
    
except ImportError:
    print("XGBoost 未安装。安装命令: pip install xgboost")


---

## 16. 进阶：时间序列交叉验证（Purged K-Fold）

> 金融数据是时间序列，不能用普通的随机 K-Fold。Purged K-Fold 在每折之间插入"清除期"（purge gap），防止训练集和验证集之间的标签重叠泄露。


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def purged_ts_cv(X, y, n_splits=5, purge_gap=10):
    """
    带清除期的时间序列交叉验证。
    purge_gap: 训练/验证之间的清除天数（防止标签重叠泄露）
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        # 清除训练集末尾 purge_gap 天
        if len(train_idx) > purge_gap:
            train_idx = train_idx[:-purge_gap]
        yield fold, train_idx, val_idx

# 使用
n_splits = 3
fold_results = []

for fold, train_idx, val_idx in purged_ts_cv(X_train, y_train, n_splits=n_splits, purge_gap=20):
    X_fold_train, y_fold_train = X_train[train_idx], y_train[train_idx]
    X_fold_val, y_fold_val = X_train[val_idx], y_train[val_idx]
    
    coef = fit_ridge(X_fold_train, y_fold_train, lam=10.0)
    pred_val = predict_ridge(X_fold_val, coef)
    ic = pd.Series(pred_val).corr(pd.Series(y_fold_val), method="spearman")
    
    fold_results.append({
        "fold": fold, "ic": ic,
        "train_size": len(train_idx), "val_size": len(val_idx),
    })

cv_results = pd.DataFrame(fold_results)
print("\nPurged TimeSeries CV 结果：")
print(cv_results.round(4))
print(f"\n平均 CV IC: {cv_results['ic'].mean():.4f} ± {cv_results['ic'].std():.4f}")


---

## 17. 进阶：过拟合诊断清单

> ML 模型在金融数据上极易过拟合。请逐项检查：


In [ ]:
# --- 诊断1：训练集 vs 测试集 IC 差距 ---
train_ic = pd.Series(pred_train).corr(pd.Series(y_train), method="spearman")
test_ic = pd.Series(pred_test).corr(pd.Series(y_test), method="spearman")
gap = train_ic - test_ic

print(f"训练集 IC: {train_ic:.4f}")
print(f"测试集 IC: {test_ic:.4f}")
print(f"IC 差距:   {gap:.4f}")
if gap > 0.05:
    print("⚠️ IC 差距过大（>0.05），模型可能过拟合")
else:
    print("✓ IC 差距在合理范围")

# --- 诊断2：特征数量 vs 样本量 ---
n_features = X_train.shape[1]
n_samples = X_train.shape[0]
ratio = n_samples / n_features
print(f"\n特征数: {n_features}, 训练样本数: {n_samples}")
print(f"样本/特征比: {ratio:.1f}")
if ratio < 10:
    print("⚠️ 样本/特征比 < 10，过拟合风险高")

# --- 诊断3：预测值分布 ---
pred_series = pd.Series(pred_test)
print(f"\n预测值标准差: {pred_series.std():.4f}")
if pred_series.std() < 0.001:
    print("⚠️ 预测值方差过小，模型可能退化为常数预测")

# --- 诊断4：分年度 IC 稳定性（建议检查） ---
print("\n建议：按年度/季度计算 IC，检查时间稳定性")


**过拟合红灯清单**：

- [ ] 训练集IC - 测试集IC > 0.05
- [ ] 样本/特征比 < 10
- [ ] CV各折IC标准差 > 均值
- [ ] 分年度IC有明显下降趋势
- [ ] 去除Top3特征后IC大幅衰减

---

本课程内容仅用于量化研究学习，不构成投资建议。
